# Build City Hourly Demand

This notebook aggregates cleaned Divvy trip-level data into a city-level hourly demand dataset.

Output metrics include total trip count, member and casual trip counts, average trip duration, and time-based flags such as weekend and rush hour.

Import and Paths

In [30]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("D:/Projects/chicago-bike-demand")
PROCESSED_DIR = BASE_DIR / "01_data" / "processed"

input_path = PROCESSED_DIR / "cleaned_divvy_trips.csv"
output_path = PROCESSED_DIR / "city_hourly_demand.csv"

### Build Function

In [31]:
def build_city_hourly_demand(input_file: Path) -> pd.DataFrame:
    df = pd.read_csv(input_file)

    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")

    city_hourly_demand = (
        df.assign(
            date=df["started_at"].dt.date,
            hour=df["started_at"].dt.hour,
            day_of_week=df["started_at"].dt.dayofweek,
        )
        .assign(
            is_weekend=lambda x: x["day_of_week"].isin([5, 6]),
            rush_hour=lambda x: x["hour"].isin([7, 8, 9, 16, 17, 18]),
        )
        .groupby(["date", "hour"], as_index=False)
        .agg(
            trip_count=("ride_id", "count"),
            member_trip_count=("member_casual", lambda x: (x == "member").sum()),
            casual_trip_count=("member_casual", lambda x: (x == "casual").sum()),
            avg_trip_duration_min=("trip_duration_min", "mean"),
            is_weekend=("is_weekend", "max"),
            rush_hour=("rush_hour", "max"),
        )
    )

    city_hourly_demand["avg_trip_duration_min"] = (
        city_hourly_demand["avg_trip_duration_min"].round(2)
    )

    return city_hourly_demand

### Transformation

In [32]:
city_hourly_demand = build_city_hourly_demand(input_path)

print("City hourly demand created successfully.")
print("Output shape:", city_hourly_demand.shape)
city_hourly_demand.head()

City hourly demand created successfully.
Output shape: (2209, 8)


,date,hour,trip_count,member_trip_count,casual_trip_count,avg_trip_duration_min,is_weekend,rush_hour
0,2025-02-28,22,1,1,0,109.84,False,False
1,2025-02-28,23,25,19,6,18.15,False,False
2,2025-03-01,0,122,69,53,8.53,True,False
3,2025-03-01,1,85,52,33,8.42,True,False
4,2025-03-01,2,46,26,20,7.60,True,False


### Validation

In [34]:
print("Missing values by column:")
print(city_hourly_demand.isna().sum())

print("\nTrip count check:")
print(
    (
        city_hourly_demand["trip_count"]
        == city_hourly_demand["member_trip_count"] + city_hourly_demand["casual_trip_count"]
    ).all()
)

print("\nUnique date-hour key check:")
print(
    city_hourly_demand[["date", "hour"]].duplicated().sum() == 0
)

print("\nHour range check:")
print(city_hourly_demand["hour"].between(0, 23).all())

Missing values by column:
date                     0
hour                     0
trip_count               0
member_trip_count        0
casual_trip_count        0
avg_trip_duration_min    0
is_weekend               0
rush_hour                0
dtype: int64

Trip count check:
True

Unique date-hour key check:
True

Hour range check:
True


### Save File

In [35]:
output_path.parent.mkdir(parents=True, exist_ok=True)
city_hourly_demand.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: D:\Projects\chicago-bike-demand\01_data\processed\city_hourly_demand.csv


In [36]:
print("Date range:")
print("Min date:", city_hourly_demand["date"].min())
print("Max date:", city_hourly_demand["date"].max())

Date range:
Min date: 2025-02-28
Max date: 2025-05-31


In [37]:
city_hourly_demand.sort_values(["date", "hour"]).head(10)

,date,hour,trip_count,member_trip_count,casual_trip_count,avg_trip_duration_min,is_weekend,rush_hour
0,2025-02-28,22,1,1,0,109.84,False,False
1,2025-02-28,23,25,19,6,18.15,False,False
2,2025-03-01,0,122,69,53,8.53,True,False
3,2025-03-01,1,85,52,33,8.42,True,False
4,2025-03-01,2,46,26,20,7.60,True,False
5,2025-03-01,3,31,17,14,6.50,True,False
6,2025-03-01,4,20,14,6,8.61,True,False
7,2025-03-01,5,18,12,6,8.19,True,False
8,2025-03-01,6,58,48,10,7.38,True,False
9,2025-03-01,7,99,83,16,7.78,True,True


In [38]:
print("Missing values by column:")
print(city_hourly_demand.isna().sum())

print("\nTrip count check:")
print(
    (
        city_hourly_demand["trip_count"]
        == city_hourly_demand["member_trip_count"] + city_hourly_demand["casual_trip_count"]
    ).all()
)

print("\nUnique date-hour key check:")
print(city_hourly_demand[["date", "hour"]].duplicated().sum() == 0)

print("\nHour range check:")
print(city_hourly_demand["hour"].between(0, 23).all())

Missing values by column:
date                     0
hour                     0
trip_count               0
member_trip_count        0
casual_trip_count        0
avg_trip_duration_min    0
is_weekend               0
rush_hour                0
dtype: int64

Trip count check:
True

Unique date-hour key check:
True

Hour range check:
True


In [39]:
print(city_hourly_demand.shape)

(2209, 8)


In [40]:
print("Missing values:")
print(city_hourly_demand.isna().sum())

print("\nTrip count check:")
print(
    (
        city_hourly_demand["trip_count"]
        == city_hourly_demand["member_trip_count"] + city_hourly_demand["casual_trip_count"]
    ).all()
)

print("\nDuplicate date-hour rows:")
print(city_hourly_demand[["date", "hour"]].duplicated().sum())

Missing values:
date                     0
hour                     0
trip_count               0
member_trip_count        0
casual_trip_count        0
avg_trip_duration_min    0
is_weekend               0
rush_hour                0
dtype: int64

Trip count check:
True

Duplicate date-hour rows:
0


### Validation Summary

- No missing values were found in the aggregated output
- Total trip count matches member plus casual trip counts for every row
- Each row represents a unique date-hour combination
- Hour values fall within the expected 0-23 range

This output is ready to be joined with external weather data for further analysis.